<a href="https://colab.research.google.com/github/changsksu/K-State_IMSE785/blob/main/KMeans_MLlib_Earthquake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
# This example is based on Bengfort k-means algorithm codes

# Install PySpark in Google Colab if needed
try:
    import pyspark
except ImportError:
    %pip install pyspark
!pip install findspark

In [25]:
# import necessary components
import os
import findspark
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# Load the dataset
import numpy as np

file_location = "drive/My Drive/Colab Notebooks/IMSE_Data_Science/Data/earthquakes.csv"

# Define parsing function
def parse_vector(line, sep=','):
    try:
        fields = line.strip().split(sep)
        latitude = float(fields[1])
        longitude = float(fields[2])
        return np.array([latitude, longitude])
    except ValueError:
        return None

# Load and parse data
data = sc.textFile(file_location).map(parse_vector)
new_data = data.filter(lambda x: x is not None)

# Split data into training and testing sets
x_train, x_test = new_data.randomSplit([0.8, 0.2], seed=42)

In [18]:
x_train.take(5)

[array([  38.8003349, -122.7981644]),
 array([  38.7939987, -122.7949982]),
 array([  38.7971649, -122.8079987]),
 array([  38.7956657, -122.7971649]),
 array([  38.8391685, -122.8433304])]

In [19]:
data.take(5)

[array([  38.8003349, -122.7981644]),
 array([  38.7939987, -122.7949982]),
 array([  38.7971649, -122.8079987]),
 array([  38.7956657, -122.7971649]),
 array([  38.8391685, -122.8433304])]

In [20]:
new_data.count()

8223

In [21]:
#
#Import packages from MLlib
from pyspark.mllib.clustering import KMeans
#
# the dataset is called data from cmd1; k is 4
clusters = KMeans.train(x_train, 4, maxIterations=10, initializationMode="random")
#
#Print the number of points within each cluster
resultRDD = data.map(lambda point: clusters.predict(point)).cache()
print("Total members of each cluster:")
print(resultRDD.countByValue())
#
#print("Cluster assignments per point:")
#print(resultRDD.collect())

Total members of each cluster:
defaultdict(<class 'int'>, {2: 5445, 0: 2076, 1: 363, 3: 339})


In [22]:
# Shows the cluster centers.
centers = clusters.clusterCenters
print("Cluster Centers: ")
for center in centers:
    print(center)

Cluster Centers: 
[  60.9695048  -152.40041646]
[ 17.13731701 -56.49704897]
[  37.29784784 -119.53741622]
[ 16.11402409 134.79177847]


In [23]:
#Evaluate clustering by computing Within Set Sum of Squared Errors
from math import sqrt
#
def error(point):
    center = clusters.centers[clusters.predict(point)]
    return sqrt(sum([x**2 for x in (point - center)]))

WSSSE = data.map(lambda point: error(point)).reduce(lambda x, y: x + y)
print("Within Set Sum of Squared Error = " + str(WSSSE))

Within Set Sum of Squared Error = 74782.91998203326


In [ ]:
#understand MLlib KMeans
# https://spark.apache.org/docs/latest/api/python/pyspark.mllib.html#pyspark.mllib.clustering.KMeans.train

In [ ]:
# your turn:
# provide a plot for the elbow method: plot WSSSE vs k and find the k*